# 构建运动发酵单胞菌酶约束模型 (eciZM564)

本笔记本基于 iZM564 基因组级代谢模型 (GEM)，结合 [ECMpy](https://github.com/tibbdc/ECMpy) 工作流与
[AutoPACMEN](https://github.com/klamt-lab/autopacmen) 酶动力学数据库，
构建 Zymomonas mobilis 的酶约束代谢模型 (ecGEM / eciZM564)，并通过迭代校准
使模型在不同培养基 (Syn / MM) 下的预测生长速率与实验值吻合。

## 目录

| 章节 | 内容 | 关键产物 |
| --- | --- | --- |
| §1 | 环境配置与依赖 | `CONFIG`, `RANDOM_SEED` |
| §2 | 工具函数 | `run_calibration_loop`, `add_sink_reactions` 等 |
| §3 | 数据预处理 (一次性) | `iZM564_reaction_gene_subunit_irr_enz.json` |
| §4 | AutoPACMEN kcat 数据库构建 (一次性) | `reaction_kcat_MW_auto.csv` |
| §5 | 计算酶分配比例 f (一次性) | `f` (≈ 0.51) |
| §6 | 构建初始 ecModel 并进行 pFBA 模拟 | `iZM564_irr_enz_constraint.json` |
| §7 | kcat 校准 (BRENDA + 手动覆盖) | `*_brenda.json` |
| §8 | 多培养基迭代拟合 | `eciZM564_*.json` |
| §9 | MM0.15 默认条件 | 注释说明 |

## 依赖 (Python >= 3.9)

- `cobra` >= 0.26
- `pandas` >= 1.5
- `numpy` >= 1.22
- `requests` (用于 UniProt API)
- 本地脚本: `script/ECMpy_function.py`, `script/AutoPACMEN_function.py`

## 复现性说明

- 路径与参数集中在 §1 的 `CONFIG` 字典中；
- 求解器固定为 `gurobi`；如不可用请回退到 `glpk`（仅限快速验证）；
- 随机种子默认 `42`；本工作流不含随机性，种子主要用于未来扩展。


In [11]:
# =============================================================================
# §1 环境配置与依赖
# =============================================================================
# 本节集中声明所有依赖、路径与可调参数，便于其他开发者快速复现。

import os
import re
import sys
import json
import random
from collections import defaultdict

import numpy as np
import pandas as pd
import cobra
from cobra.core import Reaction

# ---- 固定随机种子，确保未来引入随机算法时仍可复现 ---------------------------
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

# ---- 本地脚本路径 -----------------------------------------------------------
PROJECT_ROOT = os.path.abspath(os.getcwd())
SCRIPT_DIR = os.path.join(PROJECT_ROOT, "script")
if SCRIPT_DIR not in sys.path:
    sys.path.append(SCRIPT_DIR)

os.makedirs("./analysis", exist_ok=True)

from ECMpy_function import *
from AutoPACMEN_function import *

# ---- 求解器设置 ------------------------------------------------------------
# 默认使用 Gurobi（一致性最佳）；若无 Gurobi 可回退到 glpk，但需注意数值差异。
SOLVER = os.environ.get("ECM_SOLVER", "gurobi")
cobra.Configuration.solver = SOLVER
print(f"[CONFIG] 求解器: {SOLVER}")
print(f"[CONFIG] 随机种子: {RANDOM_SEED}")
print(f"[CONFIG] Python: {sys.version.split()[0]}, cobra: {cobra.__version__}")


[CONFIG] 求解器: gurobi
[CONFIG] 随机种子: 42
[CONFIG] Python: 3.9.2, cobra: 0.22.1


In [12]:
# =============================================================================
# §1.2 集中式配置 (CONFIG)
# =============================================================================
# 所有路径、模型参数与运行开关均集中在 CONFIG 字典中；如需在不同环境中运行，
# 仅修改本节即可。

CONFIG = {
    # --- FBA / 优化参数 ---
    "objective": "ZM_BIOMASS",
    "substrate_rxn": "EX_glc_D_e",
    "substrate_ub": 65,                      # 葡萄糖最大摄取速率 (mmol/gDW/h)
    "max_calibration_rounds": 100,            # kcat 迭代校准最大轮数

    # --- 酶约束相关参数 ----------------------------------------------------
    "ptot": 0.605,                           # 细胞总蛋白占比 (Z. mobilis) PMID: 20967753 and PMID: 21092328
    "sigma": 0.5,                            # 酶平均饱和度
    "f": 0.51,                               # 酶分配比例; §5 中可被实测值覆盖
    "f_value_list": [0.5038865319482876,
                     0.514777937088032,
                     0.5135171705402053],   # iZM564 三次生物学重复测得 f
    "lowerbound": 0,                         # 酶浓度下界

    # --- 实验目标生长速率 (用作校准阈值) ------------------------------------
    # 参考: 本研究 ZM4 在 MM 培养基的实测比生长速率
    "target_growth_rates": {
        "MM":   0.36,     # 文献 MM 实测 (中速)
        "MM015": 0.1533,  # 基本 MM 实测 (低速)
    },
    'gap_fill': 'mean',#'mean'#'median'
    'project_name': "iZM564_mean",
    'organism': "zymomonas mobilis",


    # --- 路径配置 ----------------------------------------------------------
    "paths": {
        "autopacmen_dir":  "./ZM4_get_data/",
        "data_dir":        "./data/",
        "model_dir":       "./model/",
        "analysis_dir":    "./analysis/",
        "kcat_database":   "./data/kcat_database_20260411.csv",
        "bigg_metabolites":"./data/bigg_models_metabolites.txt",
        "brenda_textfile": "./data/brenda_download.txt",
        "gem_model":       "./model/iZM564_uniprot_modification_del.json",
        "rxn_gene_subunit":"./data/iZM564_reaction_gene_subunit_irr_enz.json",
        "manual_kcat":     "./data/manual_kcat_overrides.json",
        "ec_max":          "./data/EC_kcat_max.json",
        "gene_abundance": ["./data/ZM4_1.tsv",
                           "./data/ZM4_2.tsv",
                           "./data/ZM4_3.tsv"],
        "modelgene2uniprot": "./analysis/modelgene2uniprot.txt",
        "paxdbgene2uniprot": "./analysis/paxdbgene2uniprot.txt",
    },

    # --- 输出路径 ----------------------------------------------------------
    "outputs": {
        "brenda_json":       "kcat_database_brenda.json",
        "brenda_for_model":  "kcat_database_brenda_for_model.json",
        "bigg_id_name":      "bigg_id_name_mapping.json",
        "sabio_rk":          "kcat_database_sabio_rk.json",
        "combined":          "kcat_database_combined.json",
        "reaction_kcat_MW":  "reaction_kcat_MW_auto.csv",
        "ecModel_initial":   "./model/iZM564_irr_enz_constraint.json",
        "ecModel_brenda":    "./model/iZM564_irr_enz_constraint_brenda.json",
        "kcat_brenda_csv":   "./ZM4_get_data/iZM564_mean_reaction_kcat_MW_brenda.csv",
        "rxn_kcat_out":      "./analysis/reaction_change_by_enzuse.csv",
    },

    # --- 拟合条件配置 ------------------------------------------------------
    "fitting_conditions": {
        "MM_high": {
            "ecModel_final":   "./model/eciZM564_MM_adj_final.json",
            "kcat_csv":        "./analysis/eciZM564_MM_reaction_kcat_MW_final_file.csv",
            "add_sinks":       True,
        },"MM_low": {
            "ecModel_final":   "./model/eciZM564_MM015_adj_final.json",
            "kcat_csv":        "./analysis/eciZM564_MM015_reaction_kcat_MW_final_file.csv",
            "add_sinks":       True,
        },
    },
}

# ---- 派生参数 --------------------------------------------------------------
f_measured = round(np.mean(CONFIG["f_value_list"]), 2)
CONFIG["f"] = f_measured
CONFIG["upperbound"] = round(CONFIG["ptot"] * f_measured * CONFIG["sigma"], 3)

# ---- AutoPACMEN 文件路径 (基于 autopacmen_dir 自动生成) ----------------------
AP_DIR = CONFIG["paths"]["autopacmen_dir"]
create_file(AP_DIR)
CONFIG["outputs"].update({
    "brenda_json":      os.path.join(AP_DIR, CONFIG["outputs"]["brenda_json"]),
    "brenda_for_model": os.path.join(AP_DIR, CONFIG["outputs"]["brenda_for_model"]),
    "bigg_id_name":     os.path.join(AP_DIR, CONFIG["outputs"]["bigg_id_name"]),
    "sabio_rk":         os.path.join(AP_DIR, CONFIG["outputs"]["sabio_rk"]),
    "combined":         os.path.join(AP_DIR, CONFIG["outputs"]["combined"]),
    "reaction_kcat_MW": os.path.join(AP_DIR, CONFIG["outputs"]["reaction_kcat_MW"]),
    "rxn_kcat_csv":     os.path.join(AP_DIR, "iZM564_mean_reaction_kcat_MW.csv"),
    "rxn_kcat_brenda":  os.path.join(AP_DIR, "iZM564_mean_reaction_kcat_MW_brenda.csv"),
})

# ---- EC 最大 kcat 缓存 -----------------------------------------------------
ec_max_file = CONFIG["paths"]["ec_max"]
if not os.path.exists(ec_max_file):
    generate_ec_kcat_max(CONFIG["paths"]["kcat_database"], ec_max_file)
    print(f"[CONFIG] 已生成 EC kcat max 文件: {ec_max_file}")

print("[CONFIG] 配置加载完成，关键参数:")
print(f"  - 目标函数: {CONFIG['objective']}")
print(f"  - 底物反应: {CONFIG['substrate_rxn']} (ub={CONFIG['substrate_ub']})")
print(f"  - ptot={CONFIG['ptot']}, sigma={CONFIG['sigma']}, f={CONFIG['f']}")
print(f"  - 酶上界 (upperbound) = {CONFIG['upperbound']}")


path exists
[CONFIG] 配置加载完成，关键参数:
  - 目标函数: ZM_BIOMASS
  - 底物反应: EX_glc_D_e (ub=65)
  - ptot=0.605, sigma=0.5, f=0.51
  - 酶上界 (upperbound) = 0.154


In [13]:
# =============================================================================
# §2 工具函数
# =============================================================================
# 本节定义可复用的工具函数，集中封装重复逻辑 (媒介配置、sink 反应、迭代校准)。


def configure_medium(
    model: cobra.Model,
    substrate_rxn: str,
    substrate_ub: float = 1000.0,
) -> cobra.Model:
    """关闭模型原有的所有底物摄取反应，仅开放指定的底物 (默认葡萄糖)。

    Parameters
    ----------
    model : cobra.Model
        待配置的酶约束模型。
    substrate_rxn : str
        底物反应 ID (如 'EX_glc_D_e')。
    substrate_ub : float
        底物最大摄取速率 (mmol/gDW/h)。

    Returns
    -------
    cobra.Model
        配置后的模型 (in-place 修改)。
    """
    _, substrate_id_list, _, _ = get_model_substrate_obj(model)
    for sub_id in substrate_id_list:
        if re.search(r"_reverse", sub_id):
            r_id = sub_id.split("_reverse")[0]
            model.reactions.get_by_id(sub_id).bounds = (0, 0)
            model.reactions.get_by_id(r_id).bounds = (0, 0)
        else:
            r_id = sub_id + "_reverse"
            model.reactions.get_by_id(sub_id).bounds = (0, 0)
            model.reactions.get_by_id(r_id).bounds = (0, 0)

    # 重新开放目标底物 (正向关闭，反向打开 = 摄取)
    model.reactions.get_by_id(substrate_rxn).bounds = (0, 0)
    model.reactions.get_by_id(substrate_rxn + "_reverse").bounds = (0, substrate_ub)
    return model


def add_sink_reactions(model: cobra.Model, metabolites: list) -> cobra.Model:
    """为一组代谢物添加 SK_ 开头的需求 (sink) 反应，并默认关闭摄取。

    Parameters
    ----------
    model : cobra.Model
        待修改的模型。
    metabolites : list[str]
        ModelSEED 化合物 ID 列表 (不带 _c0 后缀)。

    Returns
    -------
    cobra.Model
        修改后的模型。
    """
    for met in metabolites:
        cpd_id = met + "_c0"
        sink_rxn_id = "SK_" + cpd_id
        if sink_rxn_id in model.reactions:
            continue
        new_rxn = Reaction(sink_rxn_id)
        new_rxn.add_metabolites({model.metabolites.get_by_id(cpd_id): 1.0})
        model.add_reactions([new_rxn])
        model.reactions.get_by_id(sink_rxn_id).bounds = (0,0)
        model.medium[sink_rxn_id] = 0.0
    return model


def run_calibration_loop(
    ec_model_path: str,
    json_output_file: str,
    fluxes_outfile: str,
    ec_max_file: str,
    reaction_kcat_mw: pd.DataFrame,
    f: float,
    ptot: float,
    sigma: float,
    lowerbound: float,
    upperbound: float,
    target_growth_rate: float,
    substrate_rxn: str,
    substrate_ub: float = 1000.0,
    max_rounds: int = 50,
    add_sinks: bool = False,
    sink_metabolites: list = None,
) -> tuple:
    """迭代校准 ecGEM 直至达到目标生长速率。

    Parameters
    ----------
    ec_model_path : str
        BRENDA 校准后的 ecModel JSON 路径。
    json_output_file : str
        每次迭代后输出的 ecModel JSON 路径。
    fluxes_outfile : str
        初始 pFBA 流量文件路径。
    ec_max_file : str
        EC 最大 kcat 文件路径。
    reaction_kcat_mw : pd.DataFrame
        反应 kcat / MW 表，索引为反应 ID。
    f, ptot, sigma, lowerbound, upperbound : float
        酶约束相关参数。
    target_growth_rate : float
        实验目标生长速率 (1/h)；低于该值则继续迭代。
    substrate_rxn : str
        底物反应 ID。
    substrate_ub : float
        底物最大摄取速率。
    max_rounds : int
        最大迭代轮数 (防止无限循环)。
    add_sinks : bool
        是否添加 SK_ 反应 (用于 Syn 培养基)。
    sink_metabolites : list[str] | None
        添加 sink 反应的代谢物列表。

    Returns
    -------
    (enz_model, reaction_kcat_mw, need_change_reaction_list, changed_reaction_list)
    """
    enz_model = get_enzyme_constraint_model(ec_model_path)
    need_change_reaction_list = []
    changed_reaction_list = []
    enz_model_obj_max = 0.0

    for round_idx in range(1, max_rounds + 1):
        # 1) 通过酶使用率调整 kcat
        enz_model, reaction_kcat_mw, need_change_reaction_list, changed_reaction_list = \
            change_enz_model_by_enz_usage(
                ec_model_path, fluxes_outfile, ec_max_file,
                reaction_kcat_mw, need_change_reaction_list, changed_reaction_list,
                f, ptot, sigma, lowerbound, upperbound, json_output_file,
            )

        # 2) 配置培养基，仅开放目标底物
        enz_model = configure_medium(enz_model, substrate_rxn, substrate_ub)

        # 3) 可选：添加 sink 反应
        if add_sinks and sink_metabolites:
            enz_model = add_sink_reactions(enz_model, sink_metabolites)

        # 4) 设定目标函数并求解 (pFBA 增强数值稳定性)
        enz_model.objective = CONFIG["objective"]

        solution = cobra.flux_analysis.pfba(enz_model)
        fluxes_outfile = f'./analysis/ECMpy_solution_adjround{round_idx}.csv'
        get_fluxes_detail_in_model(enz_model, solution, fluxes_outfile, json_output_file)

        enz_model_obj_max = solution.fluxes[CONFIG["objective"]]
        enz_model_sub = -solution.fluxes[substrate_rxn + "_reverse"]
        # for key in solution.fluxes.keys():
        #     if 'EX' in key or 'SK' in key and solution.fluxes[key] > 0:
        #         print(key,solution.fluxes[key])
        print(f"[Round {round_idx:2d}] growth={enz_model_obj_max:.4f}, "
              f"glc_uptake={enz_model_sub:.2f}")

        if enz_model_obj_max >= target_growth_rate:
            print(f"[Round {round_idx}] 已达到目标生长速率 {target_growth_rate} ✓")
            break
    else:
        print(f"[WARNING] {max_rounds} 轮迭代后仍未达到目标 {target_growth_rate}")

    return enz_model, reaction_kcat_mw, need_change_reaction_list, changed_reaction_list


# ---- Syn 培养基的 sink 代谢物列表 -----------------------------------------
SYN_SINK_METABOLITES = [
    "cpd00393", "cpd00121", "cpd00218", "cpd00220",
    "cpd00128", "cpd00122", "cpd00054", "cpd00084", "cpd00161",
    "cpd00051", "cpd00039", "cpd00322", "cpd00066", "cpd00129", "cpd00069",
    "cpd00156", "cpd00060", "cpd00065", "cpd00119", "cpd00107",
]
# 注: 'cpd00041' (L-asparagine) 在 Syn 培养基中常省略，避免主竞争路径

print("[UTILS] 工具函数已定义: configure_medium, add_sink_reactions, run_calibration_loop")


[UTILS] 工具函数已定义: configure_medium, add_sink_reactions, run_calibration_loop


## §3 数据预处理 (一次性)

> 本节生成反应-基因-亚基映射文件 (`iZM564_reaction_gene_subunit_irr_enz.json`)。
> 仅需在 GEM 更新后运行一次；后续迭代拟合复用该文件。


In [14]:
model_gem = cobra.io.load_json_model(CONFIG['paths']['gem_model'])

# 1) 读取亚基数映射
with open(os.path.join(CONFIG['paths']['data_dir'], 'gene_subnum.txt')) as rf:
    subunit_count = {
        line.split('\t')[0].replace('ZMO1_', ''): line.split('\t')[-1].strip()
        for line in rf
    }

# 2) 写出原始映射
raw_map_path = os.path.join(CONFIG['paths']['data_dir'],
                             'iZM564_reaction_gene_subunit.txt')
with open(raw_map_path, 'w') as wf:
    wf.write('reaction_ID\tGPR\tsubnit_num\treaction_name\t'
             'lower_bound\tupper_bound\n')
    for rxn in model_gem.reactions:
        if not rxn.gene_reaction_rule:
            continue
        sub_count = rxn.gene_reaction_rule
        for gid, cnt in subunit_count.items():
            sub_count = sub_count.replace(gid, cnt)
        wf.write(f"{rxn.id}\t{rxn.gene_reaction_rule}\t{sub_count}\t"
                 f"{rxn.name}\t{rxn.lower_bound}\t{rxn.upper_bound}\n")

# 3) 分割双向反应 / 同工酶，并去掉 GPR 中的括号
irr_path = raw_map_path.replace('.txt', '_irr_enz.txt')
with open(raw_map_path) as rf, open(irr_path, 'w') as wf:
    wf.write('reaction_ID\tGPR\tsubnit_num\treaction_name\t'
             'lower_bound\tupper_bound\n')
    for line in rf:
        if line.startswith('reaction_ID'):
            continue
        parts = line.rstrip('\n').split('\t')
        r_id, r_gpr, r_subnum, r_name, r_lb, r_ub = parts
        r_gpr = r_gpr.replace('(', '').replace(')', '')
        lb, ub = float(r_lb), float(r_ub)

        if lb < 0 and ub > 0:
            # 双向反应: 拆分为正向/反向
            if ' or ' in r_gpr:
                for i, (g, s) in enumerate(zip(r_gpr.split(' or '),
                                                 r_subnum.split(' or ')), 1):
                    wf.write(f"{r_id}_num{i}\t{g}\t{s}\t{r_name}\t{lb}\t{ub}\n")
                    wf.write(f"{r_id}_reverse_num{i}\t{g}\t{s}\t{r_name}\t{lb}\t{ub}\n")
            else:
                wf.write(f"{r_id}\t{r_gpr}\t{r_subnum}\t{r_name}\t0\t{ub}\n")
                wf.write(f"{r_id}_reverse\t{r_gpr}\t{r_subnum}\t{r_name}\t{lb}\t0\n")
        else:
            if ' or ' in r_gpr:
                for i, (g, s) in enumerate(zip(r_gpr.split(' or '),
                                                 r_subnum.split(' or ')), 1):
                    wf.write(f"{r_id}_num{i}\t{g}\t{s}\t{r_name}\t{lb}\t{ub}\n")
            else:
                wf.write(f"{r_id}\t{r_gpr}\t{r_subnum}\t{r_name}\t{lb}\t{ub}\n")

# 4) 写出 JSON 格式 (供 §4 使用)
json_map = {}
with open(irr_path) as rf:
    for line in rf:
        if line.startswith('reaction_ID'):
            continue
        r_id, r_gpr, r_subnum = line.rstrip('\n').split('\t')[:3]
        r_gpr   = r_gpr.replace('(', '').replace(')', '')
        r_subnum = r_subnum.replace('(', '').replace(')', '')
        entry = {}
        if ' and ' in r_gpr:
            for g, s in zip(r_gpr.split(' and '), r_subnum.split(' and ')):
                entry[g.strip()] = s.strip()
        else:
            entry[r_gpr.strip()] = r_subnum.strip()
        json_map[r_id] = entry

with open(CONFIG['paths']['rxn_gene_subunit'], 'w') as wf:
    json.dump(json_map, wf, indent=2)

print(f"[§3] 已生成 {CONFIG['paths']['rxn_gene_subunit']}，"
      f"含 {len(json_map)} 个反应")


[§3] 已生成 ./data/iZM564_reaction_gene_subunit_irr_enz.json，含 2129 个反应


## §4 AutoPACMEN kcat 数据库构建 (一次性)

> 参考 [klamt-lab/autopacmen](https://github.com/klamt-lab/autopacmen)。
> 该流程会从 BRENDA / SABIO-RK / UniProt 下载并合并数据，耗时较长，
> 因此产物可缓存复用。仅当 BRENDA 数据库或 GEM 更新时需要重跑。


In [ ]:
# §4.1 AutoPACMEN 完整流水线 (按需运行)
# 取消下方注释即可执行；首次运行可能耗时数十分钟到数小时。

# P = CONFIG['paths']
# O = CONFIG['outputs']

# # Step 1: BIGG metabolites
# print('Step 1/8: BIGG metabolites...')
# parse_bigg_metabolites_file(P['bigg_metabolites'], P['autopacmen_dir'])

# # Step 2: BRENDA kcat
# print('Step 2/8: BRENDA kcat...')
# parse_brenda_textfile(P['brenda_textfile'], P['autopacmen_dir'], O['brenda_json'])

# # Step 3: BRENDA → model
# print('Step 3/8: BRENDA kcat for model...')
# parse_brenda_json_for_model(P['gem_model'], O['brenda_json'], O['brenda_for_model'])

# # Step 4: SABIO-RK kcat
# print('Step 4/8: SABIO-RK kcat...')
# parse_sabio_rk_for_model(P['gem_model'], O['sabio_rk'], O['bigg_id_name'])

# # Step 5: 合并 BRENDA + SABIO-RK
# print('Step 5/8: Combined kcat database...')
# create_combined_kcat_database(O['sabio_rk'], O['brenda_for_model'], O['combined'])

# # Step 6: 反应-基因 → kcat 映射
# print('Step 6/8: Reactions-kcat mapping...')
# get_reactions_kcat_mapping(
#     P['gem_model'], P['autopacmen_dir'], CONFIG['project_name'],
#     CONFIG['organism'], O['combined'], 'none', CONFIG['gap_fill'],
# )

# # Step 7: UniProt → 蛋白分子量
# print('Step 7/8: UniProt protein mass mapping...')
# get_protein_mass_mapping(P['gem_model'], P['autopacmen_dir'], CONFIG['project_name'])

# # Step 8: 反应级 kcat / MW
# print('Step 8/8: Reaction kcat_MW table...')
# get_reaction_kcat_mw(CONFIG['paths']['gem_model'], CONFIG['paths']['autopacmen_dir'], CONFIG['project_name'],CONFIG['paths']['rxn_gene_subunit'], CONFIG['gap_fill'])

# print(f"[§4] AutoPACMEN 产物: {O['reaction_kcat_MW']}")


## §5 计算酶分配比例 f (一次性)

> 通过比较模型基因 vs 全蛋白质组的 UniProt 映射丰度，得到酶占细胞总蛋白
> 的比例 `f`。本研究中三次生物学重复得到 f ≈ 0.51。


In [16]:
# §5.1 计算 f (按需运行)
# 注: 本结果已缓存于 CONFIG['f_value_list']，通常无需重跑。

# from ECMpy_function import GENENAME_2_ACC_from_uniprot, calculate_f_special

# model = cobra.io.load_json_model(CONFIG['paths']['gem_model'])
# model_gene_list = [g.id.replace('ZMO1_', '') for g in model.genes]
# GENENAME_2_ACC_from_uniprot(
#     list(set(model_gene_list)), CONFIG['paths']['modelgene2uniprot']
# )

# f_values = []
# for abundance_file in CONFIG['paths']['gene_abundance']:
#     GENENAME_2_ACC_from_uniprot(
#         pd.read_csv(abundance_file, sep='\t', index_col='gene_id').index.tolist(),
#         CONFIG['paths']['paxdbgene2uniprot'],
#     )
#     f = calculate_f_special(
#         abundance_file,
#         CONFIG['paths']['modelgene2uniprot'],
#         CONFIG['paths']['paxdbgene2uniprot'],
#         'normalized_counts',
#     )
#     f_values.append(f)
# print(f"f = {f_values}, mean = {np.mean(f_values):.4f}")

# 显示已缓存的 f 值
f_value_list = CONFIG['f_value_list']
f_measured = round(np.mean(f_value_list), 2)
print(f"已缓存 f (3 次重复): {f_value_list}")
print(f"f (取均值, 2 位小数) = {f_measured}")


已缓存 f (3 次重复): [0.5038865319482876, 0.514777937088032, 0.5135171705402053]
f (取均值, 2 位小数) = 0.51


## §6 构建初始 ecModel 并进行 pFBA 模拟

> 利用 §4 产物的 `reaction_kcat_MW.csv` 与 §5 的 `f`，调用
> `trans_model2enz_json_model_split_isoenzyme` 将 GEM 升级为 ecModel，
> 并运行基线 pFBA 检验。


In [17]:
# §6.1 构建初始 ecModel
reaction_kcat_MW_file = os.path.join(
    CONFIG['paths']['autopacmen_dir'], 'reaction_kcat_MW.csv'
)
# 如产物文件名因 gap_fill 不同 (mean/median)，可在此调整
# 常见形式: reaction_kcat_MW.csv, iZM564_mean_reaction_kcat_MW.csv
candidate_paths = [
    reaction_kcat_MW_file,
    os.path.join(CONFIG['paths']['autopacmen_dir'],
                 f"iZM564_{CONFIG.get('gap_fill', 'mean')}_reaction_kcat_MW.csv"),
]
for p in candidate_paths:
    if os.path.exists(p):
        reaction_kcat_MW_file = p
        break
print(f"[§6] 使用 reaction_kcat_MW 文件: {reaction_kcat_MW_file}")

# 写入酶约束 (使用 CONFIG 中的 f, ptot, sigma, lowerbound, upperbound)
trans_model2enz_json_model_split_isoenzyme(
    CONFIG['paths']['gem_model'],
    reaction_kcat_MW_file,
    CONFIG['f'],
    CONFIG['ptot'],
    CONFIG['sigma'],
    CONFIG['lowerbound'],
    CONFIG['upperbound'],
    CONFIG['outputs']['ecModel_initial'],
)
print(f"[§6] 已生成初始 ecModel: {CONFIG['outputs']['ecModel_initial']}")


[§6] 使用 reaction_kcat_MW 文件: ./ZM4_get_data/reaction_kcat_MW.csv
[§6] 已生成初始 ecModel: ./model/iZM564_irr_enz_constraint.json


In [18]:
# §6.2 基线 pFBA 检验
ec_model_path = CONFIG['outputs']['ecModel_initial']
enz_model = get_enzyme_constraint_model(ec_model_path)
enz_model.objective = CONFIG['objective']

# 同步写出 CSV 供下游使用
rxn_kcat_csv_path = os.path.join(
    CONFIG['paths']['autopacmen_dir'], 'iZM564_mean_reaction_kcat_MW.csv'
)
with open(rxn_kcat_csv_path, 'w') as wf:
    wf.write(',kcat,MW,kcat_MW\n')
    for rxn in enz_model.reactions:
        if rxn.kcat:
            kcat_mw = float(rxn.kcat) * 3600000 / float(rxn.kcat_MW)
            wf.write(f"{rxn.id},{rxn.kcat},{kcat_mw},{rxn.kcat_MW}\n")
print(f"[§6] 已导出 kcat_MW CSV: {rxn_kcat_csv_path}")

# 关闭默认底物，仅开放葡萄糖
enz_model = configure_medium(enz_model, CONFIG['substrate_rxn'],
                              CONFIG['substrate_ub'])

baseline_solution = cobra.flux_analysis.pfba(enz_model)
fluxes_outfile = os.path.join(
    CONFIG['paths']['analysis_dir'],
    f"ECMpy_solution_{CONFIG['objective']}_pfba.csv",
)
get_fluxes_detail_in_model(enz_model, baseline_solution, fluxes_outfile, ec_model_path)
print(f"[§6] 基线生长速率 = {baseline_solution.fluxes[CONFIG['objective']]:.4f}")
print(f"[§6] 流量文件: {fluxes_outfile}")

# 打印显著流量 (|v| > 1e-4)
significant = baseline_solution.fluxes[abs(baseline_solution.fluxes) > 1e-4]
for rxn_id, flux in significant.items():
    rxn_str = enz_model.reactions.get_by_id(rxn_id).build_reaction_string(
        use_metabolite_names=True)
    print(f"  {rxn_id:30s} {flux:>10.4f}  {rxn_str}")


[§6] 已导出 kcat_MW CSV: ./ZM4_get_data/iZM564_mean_reaction_kcat_MW.csv
Enzyme cost total is:0.14854904052121073
[§6] 基线生长速率 = 0.1128
[§6] 流量文件: ./analysis/ECMpy_solution_ZM_BIOMASS_pfba.csv
  ACTNtex                            0.3737  (S)-Acetoin --> (S)-Acetoin
  ACTNtpp                            0.3737  (S)-Acetoin --> (S)-Acetoin
  ATPM                               3.7100  H2O + ATP --> ADP + Phosphate + H+
  EX_ac_e                            0.0164  Acetate --> 
  EX_actn__S_e                       0.3737  (S)-Acetoin --> 
  EX_co2_e                           0.3780  CO2 --> 
  EX_duri_e                          0.0496  Deoxyuridine --> 
  EX_for_e                           0.2818  Formate --> 
  EX_h_e                            34.3984  H+ --> 
  EX_h2o_e                           5.7495  H2O --> 
  EX_lac_D_e                        36.0764  D-Lactate --> 
  EX_succ_e                          0.0711  Succinate --> 
  HPN                                0.0040  H2O + Bacteriohopa

## §7 kcat 校准 (BRENDA + 手动覆盖)

> 两步校准:
> 1. 从 BRENDA 合并库中提取 Z. mobilis 相关最大 kcat，按 EC 号覆盖
> 2. 应用 `manual_kcat_overrides.json` 中的经验/文献值，处理 BRENDA 未覆盖的反应
>
> 产物 `*_brenda.json` 是 §8 迭代拟合的输入。


In [19]:
# §7.1 提取 Z. mobilis 相关最大 kcat (按 EC 号汇总)
organism = 'zymomonas mobilis'
brenda_data = json.load(open(CONFIG['outputs']['combined'], encoding='utf-8'))

# 按 EC 号汇总: {ec_number: [kcat1, kcat2, ...]}
ec_kcat_max = defaultdict(list)
for ec, species_data in brenda_data.items():
    if not isinstance(species_data, dict):
        continue
    for _, org_data in species_data.items():
        if not isinstance(org_data, dict):
            continue
        for org_name, kcat_values in org_data.items():
            if organism in org_name.lower():
                for kcat in kcat_values:
                    if isinstance(kcat, (int, float)) and kcat > 0:
                        ec_kcat_max[ec].append(kcat)

# 打印摘要
print(f"{'EC 号':<15} {'Z. mobilis 最大 kcat (s^-1)':<20}")
print('-' * 35)
for ec in sorted(ec_kcat_max):
    print(f"{ec:<15} {max(ec_kcat_max[ec]):>15.2f}")
print(f"\n共 {len(ec_kcat_max)} 个 EC 号覆盖")


EC 号            Z. mobilis 最大 kcat (s^-1)
-----------------------------------
1.1.1.41                 112.00
1.1.1.49                 305.00
1.1.99.28                999.00
1.5.1.2                   27.00
2.7.2.3                  999.00
3.2.1.26               50000.00
4.1.1.1                  486.00

共 7 个 EC 号覆盖


In [20]:
# §7.2 应用 BRENDA + 手动 kcat 覆盖
# 加载手动覆盖字典
with open(CONFIG['paths']['manual_kcat'], encoding='utf-8') as f:
    raw = json.load(f)
# 过滤掉以下划线开头的元数据键
manual_kcat_overrides = {k: v for k, v in raw.items() if not k.startswith('_')}
print(f"[§7] 手动覆盖字典: {len(manual_kcat_overrides)} 个反应")

# 读取 kcat_MW 表 (用于在缺少 kcat_MW 时回推)
kcat_mw_table = pd.read_csv(
    os.path.join(CONFIG['paths']['autopacmen_dir'],
                 'iZM564_mean_reaction_kcat_MW.csv'),
    index_col=0,
)

# 读取 ecModel JSON 进行 in-place 修改
ec_model_dict = json.load(open(CONFIG['outputs']['ecModel_initial'], encoding='utf-8'))
modified_rxn_ids = set()

for rxn in ec_model_dict['reactions']:
    rxn_id = rxn['id']
    rxn_base = rxn_id.split('_')[0]   # 处理 _num / _reverse 后缀

    # --- (a) BRENDA 覆盖: 按 ec-code 匹配 ---
    ec_code = (rxn.get('annotation') or {}).get('ec-code')
    if 'ec-code' in rxn['annotation']:
        for ec in ec_kcat_max.keys():
            if ec == rxn['annotation']['ec-code']:
                max_kcat = max(ec_kcat_max[ec])
                rxn['kcat'] = max_kcat
                rxn['kcat_MW'] = max_kcat * 3600000 / kcat_mw_table.loc[rxn_id, 'MW']
                modified_rxn_ids.add(rxn_id)

    # --- (b) 手动覆盖: 优先按完整 ID 匹配，再按前缀匹配 ---
    override = None
    if rxn_id in manual_kcat_overrides:
        override = manual_kcat_overrides[rxn_id]
    elif rxn_base in manual_kcat_overrides:
        override = manual_kcat_overrides[rxn_base]

    if override is not None:
        rxn['kcat'] = override['kcat']
        if 'kcat_MW' in override:
            rxn['kcat_MW'] = override['kcat_MW']
        elif rxn_id in kcat_mw_table.index:
            rxn['kcat_MW'] = override['kcat'] * 3600000 / kcat_mw_table.loc[rxn_id, 'MW']
        modified_rxn_ids.add(rxn_id)

print(f"[§7] 累计覆盖反应数: {len(modified_rxn_ids)}")

# 写出 BRENDA 校准后的 ecModel
with open(CONFIG['outputs']['ecModel_brenda'], 'w', encoding='utf-8') as f:
    json.dump(ec_model_dict, f, ensure_ascii=False, indent=2)

# 同步更新 kcat_MW CSV
brenda_csv_path = CONFIG['outputs']['rxn_kcat_brenda']
with open(brenda_csv_path, 'w', encoding='utf-8') as wf:
    wf.write('reaction_id,kcat,MW,kcat_MW\n')
    for rxn in ec_model_dict['reactions']:
        if rxn.get('kcat_MW') and rxn['id'] in kcat_mw_table.index:
            mw = kcat_mw_table.loc[rxn['id'], 'MW']
            wf.write(f"{rxn['id']},{rxn['kcat']},{mw},{rxn['kcat_MW']}\n")
print(f"[§7] BRENDA 校准完成: {CONFIG['outputs']['ecModel_brenda']}")


[§7] 手动覆盖字典: 44 个反应
[§7] 累计覆盖反应数: 43
[§7] BRENDA 校准完成: ./model/iZM564_irr_enz_constraint_brenda.json


## §8 多培养基迭代拟合

> 调用 §2 的 `run_calibration_loop`，对不同培养基配置分别校准：
> - **MM_high**: 目标生长速率 0.36，启用 sink 反应
> - **MM015**: 默认酶约束已满足 0.1533，不启用 sink 反应
>
> 每个条件独立生成最终 ecModel JSON 与 kcat_MW 表。


In [21]:
# §8.1 MM 培养基迭代拟合 (目标生长速率 = 0.36)
cond = CONFIG['fitting_conditions']['MM_high']
target_growth = CONFIG['target_growth_rates']['MM']
print(f"[§8.MM_high] 目标生长速率: {target_growth}, 启用 sink: {cond['add_sinks']}")

rxn_kcat_mw_mm = pd.read_csv(CONFIG['outputs']['rxn_kcat_brenda'], index_col=0)

enz_model_mm, rxn_kcat_mw_mm, _, changed_mm = run_calibration_loop(
    ec_model_path=CONFIG['outputs']['ecModel_brenda'],
    json_output_file=CONFIG['outputs']['ecModel_brenda'].replace('.json', '_iter.json'),
    fluxes_outfile=fluxes_outfile,
    ec_max_file=CONFIG['paths']['ec_max'],
    reaction_kcat_mw=rxn_kcat_mw_mm,
    f=CONFIG['f'],
    ptot=CONFIG['ptot'],
    sigma=CONFIG['sigma'],
    lowerbound=CONFIG['lowerbound'],
    upperbound=CONFIG['upperbound'],
    target_growth_rate=target_growth,
    substrate_rxn=CONFIG['substrate_rxn'],
    substrate_ub=1000,
    max_rounds=CONFIG['max_calibration_rounds'],
    add_sinks=cond['add_sinks'],
    sink_metabolites=SYN_SINK_METABOLITES,
)

rxn_kcat_mw_mm.to_csv(CONFIG['outputs']['rxn_kcat_out'])
rxn_kcat_mw_mm = pd.read_csv(CONFIG['outputs']['rxn_kcat_out'], index_col=0)
rxn_kcat_mw_mm.to_csv(cond['kcat_csv'])

trans_model2enz_json_model_split_isoenzyme(
    CONFIG['paths']['gem_model'],
    cond['kcat_csv'],
    CONFIG['f'], CONFIG['ptot'], CONFIG['sigma'],
    CONFIG['lowerbound'], CONFIG['upperbound'],
    cond['ecModel_final'],
)
print(f"[§8.MM_high] 最终 ecModel: {cond['ecModel_final']}")
print(f"[§8.MM_high] kcat_MW 表: {cond['kcat_csv']}")


[§8.MM_high] 目标生长速率: 0.36, 启用 sink: True
Need changing reaction: 
rxn00459
Changed reaction: 
['rxn00459']
Enzyme cost total is:0.14854904052120832
Enzyme cost total is:0.14690419033327684
[Round  1] growth=0.1564, glc_uptake=-25.12
Need changing reaction: 
rxn05347
Changed reaction: 
['rxn00459', 'rxn05347']
Enzyme cost total is:0.1469041903332744
Enzyme cost total is:0.14566504185356838
[Round  2] growth=0.1845, glc_uptake=-28.97
Need changing reaction: 
rxn05346
Changed reaction: 
['rxn00459', 'rxn05347', 'rxn05346']
Enzyme cost total is:0.14566504185356635
Enzyme cost total is:0.14393035244690117
[Round  3] growth=0.2239, glc_uptake=-34.36
Need changing reaction: 
rxn05343
Changed reaction: 
['rxn00459', 'rxn05347', 'rxn05346', 'rxn05343']
Enzyme cost total is:0.1439303524468986
Enzyme cost total is:0.14146792052570978
[Round  4] growth=0.2626, glc_uptake=-39.65
Need changing reaction: 
rxn00781
Changed reaction: 
['rxn00459', 'rxn05347', 'rxn05346', 'rxn05343', 'rxn00781']
Enzyme 

## §9 MM0.15 默认条件

> 基本 MM 培养基下的实测比生长速率 (0.1533) .


In [22]:
# §8.2 MM 培养基迭代拟合 (目标生长速率 = 0.36)
cond = CONFIG['fitting_conditions']['MM_low']
target_growth = CONFIG['target_growth_rates']['MM015']
print(f"[§9.MM_low] 目标生长速率: {target_growth}, 启用 sink: {cond['add_sinks']}")

rxn_kcat_mw_mm = pd.read_csv(CONFIG['outputs']['rxn_kcat_brenda'], index_col=0)

enz_model_mm, rxn_kcat_mw_mm, _, changed_mm = run_calibration_loop(
    ec_model_path=CONFIG['outputs']['ecModel_brenda'],
    json_output_file=CONFIG['outputs']['ecModel_brenda'].replace('.json', '_iter.json'),
    fluxes_outfile=fluxes_outfile,
    ec_max_file=CONFIG['paths']['ec_max'],
    reaction_kcat_mw=rxn_kcat_mw_mm,
    f=CONFIG['f'],
    ptot=CONFIG['ptot'],
    sigma=CONFIG['sigma'],
    lowerbound=CONFIG['lowerbound'],
    upperbound=CONFIG['upperbound'],
    target_growth_rate=target_growth,
    substrate_rxn=CONFIG['substrate_rxn'],
    substrate_ub=1000,
    max_rounds=CONFIG['max_calibration_rounds'],
    add_sinks=cond['add_sinks'],
    sink_metabolites=SYN_SINK_METABOLITES,
)

rxn_kcat_mw_mm.to_csv(CONFIG['outputs']['rxn_kcat_out'])
rxn_kcat_mw_mm = pd.read_csv(CONFIG['outputs']['rxn_kcat_out'], index_col=0)
rxn_kcat_mw_mm.to_csv(cond['kcat_csv'])

trans_model2enz_json_model_split_isoenzyme(
    CONFIG['paths']['gem_model'],
    cond['kcat_csv'],
    CONFIG['f'], CONFIG['ptot'], CONFIG['sigma'],
    CONFIG['lowerbound'], CONFIG['upperbound'],
    cond['ecModel_final'],
)
print(f"[§9.MM_low] 最终 ecModel: {cond['ecModel_final']}")
print(f"[§9.MM_low] kcat_MW 表: {cond['kcat_csv']}")


[§9.MM_low] 目标生长速率: 0.1533, 启用 sink: True
Need changing reaction: 
rxn00459
Changed reaction: 
['rxn00459']
Enzyme cost total is:0.14854904052120832
Enzyme cost total is:0.14690419033327684
[Round  1] growth=0.1564, glc_uptake=-25.12
[Round 1] 已达到目标生长速率 0.1533 ✓
[§9.MM_low] 最终 ecModel: ./model/eciZM564_MM015_adj_final.json
[§9.MM_low] kcat_MW 表: ./analysis/eciZM564_MM015_reaction_kcat_MW_final_file.csv
